# Using Alphafold components on Google Colab

---



In [1]:
#@title Run to install dependencies
#@markdown * Copied from the [Alphafold Colab notebook](https://colab.research.google.com/github/deepmind/alphafold/blob/main/notebooks/AlphaFold.ipynb)

from IPython.utils import io
import os
import subprocess
import tqdm.notebook

TQDM_BAR_FORMAT = '{l_bar}{bar}| {n_fmt}/{total_fmt} [elapsed: {elapsed} remaining: {remaining}]'

GIT_REPO = 'https://github.com/deepmind/alphafold'
DATA_REPO = 'https://github.com/sameerd/data_dump'

force_reinstall_dependencies = False  #@param {type:"boolean"}

if (not os.path.exists("INSTALLED_DEPS")) or force_reinstall_dependencies:
  try:
    with tqdm.notebook.tqdm(total=100, bar_format=TQDM_BAR_FORMAT) as pbar:
      with io.capture_output() as captured:
        %shell rm -f INSTALLED_DEPS
        # Uninstall default Colab version of TF.
        %shell pip uninstall -y tensorflow
        pbar.update(6)
        %shell rm -rf alphafold
        %shell git clone --branch main {GIT_REPO} alphafold
        pbar.update(8)
        %shell pip3 install -r ./alphafold/requirements.txt
        pbar.update(50)
        # Run setup.py to install only AlphaFold.
        %shell pip3 install --no-dependencies ./alphafold
        pbar.update(28)
        # add optax in case we want to use an optimizer
        %shell pip3 install --no-dependencies optax
        pbar.update(4)
        %shell rm -rf data_dump
        %shell git clone --branch main {DATA_REPO} data_dump
        pbar.update(4)
        %shell touch INSTALLED_DEPS
  except subprocess.CalledProcessError:
    print(captured)
    raise


import jax
if jax.local_devices()[0].platform == 'tpu':
  raise RuntimeError('Colab TPU runtime not supported. Change it to GPU via Runtime -> Change Runtime Type -> Hardware accelerator -> GPU.')
elif jax.local_devices()[0].platform == 'cpu':
  #raise RuntimeError('Colab CPU runtime not supported. Change it to GPU via Runtime -> Change Runtime Type -> Hardware accelerator -> GPU.')
  print(f"{jax.local_devices()}")
else:
  print(f'Running with {jax.local_devices()[0].device_kind} GPU')


# Make sure all necessary environment variables are set.
import os
os.environ['TF_FORCE_UNIFIED_MEMORY'] = '1'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '2.0'





[CpuDevice(id=0)]


In [2]:
import dataclasses
import gzip
import numpy as np
import alphafold

import alphafold.common
from alphafold.common.protein import from_pdb_string
from alphafold.common.residue_constants import restypes

In [3]:
def convert_aatype_to_string(prot_aatype : np.ndarray):
  # convert amino acid indices in protein to a sequence string
  return "".join(restypes[i] for i in prot_aatype)

with gzip.open("./data_dump/starting.pdb.gz", "rt") as fh:
    prot = from_pdb_string(fh.read())
    starting_seq = convert_aatype_to_string(prot.aatype)

print(f"Seq : {starting_seq}")
prot.__annotations__

Seq : MQHTYPAQLMRFGTAARAEHMTIAAAIHALDADEADAIVMDIVPDGERDAWWDDEGFSSSPFTKNAHHAGIVATSVTLGQLQREQGDKLVSKAAEYFGIACRVNDGLRTTRFVRLFSDALDAKPLTIGHDYEVEFLLATRRVYEPFEAPFNFAPHCDDVSYGRDTVNWPLKRSFPRQLGGFLTIQGADNDAGMVMWDNRPESRAALDEMHAEYRETGAIAALERAAKIMLKPQPGQLTLFQSKNLHAIERCTSTRRTMGLFLIHTEDGWRMFD


{'aatype': numpy.ndarray,
 'atom_mask': numpy.ndarray,
 'atom_positions': numpy.ndarray,
 'b_factors': numpy.ndarray,
 'chain_index': numpy.ndarray,
 'residue_index': numpy.ndarray}

In [4]:
print(prot.atom_positions.shape)
prot.atom_positions

(273, 37, 3)


array([[[-40.47900009,  18.67099953, -24.95700073],
        [-40.76399994,  18.16799927, -23.57099915],
        [-40.06900024,  16.84399986, -23.38299942],
        ...,
        [  0.        ,   0.        ,   0.        ],
        [  0.        ,   0.        ,   0.        ],
        [  0.        ,   0.        ,   0.        ]],

       [[-39.38000107,  16.73399925, -22.25799942],
        [-38.65000153,  15.54899979, -21.84399986],
        [-39.60800171,  14.58399963, -21.22200012],
        ...,
        [  0.        ,   0.        ,   0.        ],
        [  0.        ,   0.        ,   0.        ],
        [  0.        ,   0.        ,   0.        ]],

       [[-39.36100006,  13.30000019, -21.38500023],
        [-40.18799973,  12.31799984, -20.70899963],
        [-39.91699982,  12.22200012, -19.23999977],
        ...,
        [  0.        ,   0.        ,   0.        ],
        [  0.        ,   0.        ,   0.        ],
        [  0.        ,   0.        ,   0.        ]],

       ...,

      

## Model

In [5]:
from alphafold.model.common_modules import Linear

import numpy as np
import jax
import jax.numpy as jnp
import haiku as hk

In [6]:
def _f(x):
    m = Linear(num_output=2, num_input_dims=1, name="linear22")
    return m(x)
f = hk.transform(_f)
f

Transformed(init=<function without_state.<locals>.init_fn at 0x7ffb5595a200>, apply=<function without_state.<locals>.apply_fn at 0x7ffb5595a290>)

In [7]:
test_inp = prot.atom_positions.astype(jnp.float32)

In [8]:
rng_key = jax.random.PRNGKey(42)
params = f.init(x=test_inp, rng=rng_key)
params

FlatMapping({
  'linear22': FlatMapping({
                'weights': DeviceArray([[-0.34503725,  0.582498  ],
                                        [ 0.0910517 , -0.35958534],
                                        [ 0.68855035,  0.66172487]], dtype=float32),
                'bias': DeviceArray([0., 0.], dtype=float32),
              }),
})

In [9]:
prot.atom_positions.shape

(273, 37, 3)

In [10]:
output1 = f.apply(params, rng=rng_key, x=test_inp)
output1

DeviceArray([[[ -1.5173626 , -46.807423  ],
              [ -0.51049376, -45.875412  ],
              [ -0.74139994, -44.870083  ],
              ...,
              [  0.        ,   0.        ],
              [  0.        ,   0.        ],
              [  0.        ,   0.        ]],

             [[ -0.21452728, -43.684746  ],
              [ -0.28924042, -42.55946   ],
              [  0.38171828, -42.358902  ],
              ...,
              [  0.        ,   0.        ],
              [  0.        ,   0.        ],
              [  0.        ,   0.        ]],

             [[  0.0673494 , -41.861176  ],
              [  0.7287432 , -41.54246   ],
              [  1.6379772 , -40.37801   ],
              ...,
              [  0.        ,   0.        ],
              [  0.        ,   0.        ],
              [  0.        ,   0.        ]],

             ...,

             [[ -6.134639  , -31.984861  ],
              [ -6.269721  , -31.408993  ],
              [ -6.9840364 , -30.61801

In [11]:
from alphafold.model import config

In [12]:
model_name = ('model_1')
cfg = config.model_config(model_name)
cfg.data.eval.num_ensemble = 1

In [13]:
list(cfg.keys())

['data', 'model']

In [14]:
cfg.model.global_config

deterministic: false
multimer_mode: false
subbatch_size: 4
use_remat: false
zero_init: true

In [15]:
# make features
from alphafold.data import pipeline

In [16]:
from typing import Any, Mapping, Optional, Union

import logging
from alphafold.model.folding import InvariantPointAttention
from alphafold.model import features
import tensorflow.compat.v1 as tf

In [17]:
class RunTestModel:
  """ Container for Jax model. """

  def __init__(self, 
               config,
               params: Optional[Mapping[str, Mapping[str, np.ndarray]]] = None):
    self.config = config

    def _forward_fn(batch):
      model = InvariantPointAttention(config=self.config.model, 
                  global_config=self.config.model.global_config)
      return model(batch, is_training=True, compute_loss=True)
    self.apply = jax.jit(hk.transform(_forward_fn).apply)
    self.init = jax.jit(hk.transform(_forward_fn).init)

    def init_params(self, feat: features.FeatureDict, random_seed: int = 0):
      """Initializes the model parameters.
      If none were provided when this class was instantiated then the parameters
      are randomly initialized.
      Args:
        feat: A dictionary of NumPy feature arrays as output by
          RunModel.process_features.
        random_seed: A random seed to use to initialize the parameters if none
          were set when this class was initialized.
      """
      if not self.params:
        # Init params randomly.
        rng = jax.random.PRNGKey(random_seed)
        self.params = hk.data_structures.to_mutable_dict(
            self.init(rng, feat))
        logging.warning('Initialized parameters randomly')

In [18]:
model_runner = RunTestModel(cfg)